In [14]:
import os
import json
import pandas as pd
import traceback
import PyPDF2
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableConfig
from langchain_core.output_parsers import StrOutputParser
from langchain_community.callbacks.manager import get_openai_callback

from dotenv import load_dotenv
load_dotenv()

True

In [15]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY is missing. Add it to your .env file or environment "
        "before running this notebook."
    )

In [16]:
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url="https://openrouter.ai/api/v1", 
    model="openai/gpt-oss-120b",
    temperature=0.5,
    verbose=True
)

In [17]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    }, 
}

In [18]:
TEMPLATE="""
Text: {text}
You are an expert MCQ maker. Given the above text, it is your job to create a quiz of 
{number} multiple choice questions for {subject} students in {tone} tone. Make sure the
questions are not repeated and check all the questions to be confirming the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}
"""

In [19]:
quiz_generation_prompt = PromptTemplate(
    input_variables=["text", "number", "subject", "tone", "response_json"],
    template=TEMPLATE
)

In [20]:
quiz_chain = quiz_generation_prompt | llm | StrOutputParser()

In [21]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject}
students. You need to evaluate the complexity of the question and give a complete analysis of
the quiz. Only use at max 50 words for complexity if the quiz is not at per with the cognitive 
and analytical abilities of the students. Update the quiz questions which needs to be changed
and change the tone such that it perfectly fits the students' ability.
Quiz MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [22]:
quiz_evaluation_prompt=PromptTemplate(
    input_variables=["subject", "quiz"],
    template=TEMPLATE2
)

In [23]:
review_chain = quiz_evaluation_prompt | llm | StrOutputParser()

In [24]:
generate_evaluate_chain = (
    RunnablePassthrough()
    .assign(quiz=quiz_chain)
    .assign(review=review_chain)
)

In [25]:
file_path = Path("/workspaces/MCQ_Generator/data.txt")

if not file_path.is_file():
    local_candidate = Path("data.txt")
    if local_candidate.is_file():
        file_path = local_candidate
    else:
        raise FileNotFoundError(
            f"Could not find the input text file at {file_path} or "
            f"{local_candidate.resolve()}. Update file_path to the location of data.txt."
        )

TEXT = file_path.read_text(encoding="utf-8")

NUMBER = 5
SUBJECT = "machine learning"
TONE = "simple"

In [26]:
response = generate_evaluate_chain.invoke(
    {
        "text": TEXT,
        "number": NUMBER,
        "subject": SUBJECT,
        "tone": TONE,
        "response_json": json.dumps(RESPONSE_JSON, ensure_ascii=False),
    }
)

print("QUIZ")
print(response["quiz"])
print("\\nREVIEW")
print(response["review"])


OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}